In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import pickle

print("1. Loading dataset...")
# Load the Kaggle dataset
df = pd.read_csv('TMDB_movie_dataset_v11.csv')

print(f"Initial shape: {df.shape}")

# 2. Basic Cleaning & Quality Filtering
# Optional: Filter out obscure movies with 0 votes or no overview to keep quality high and search fast
df = df.dropna(subset=['title', 'overview'])
df = df[df['vote_count'] >= 5]  # Keeps movies with at least some user engagement
df = df.reset_index(drop=True)

print(f"Shape after filtering quality rows: {df.shape}")

# 3. Create the 'tags' feature
# Combining title, genres, keywords, and overview
df['genres'] = df['genres'].fillna('').astype(str).apply(lambda x: x.replace(',', ' '))
df['keywords'] = df['keywords'].fillna('').astype(str).apply(lambda x: x.replace(',', ' '))
df['overview'] = df['overview'].fillna('').astype(str)

df['tags'] = df['overview'] + ' ' + df['genres'] + ' ' + df['keywords']

# Keep only necessary columns to save disk space and RAM
movies_cleaned = df[['id', 'title', 'tags', 'poster_path']].rename(columns={'id': 'movie_id'})

print("4. Vectorizing tags using TF-IDF...")
# Using max_features limits vocabulary size so memory doesn't explode
vectorizer = TfidfVectorizer(max_features=25000, stop_words='english', dtype=np.float32)
tfidf_matrix = vectorizer.fit_transform(movies_cleaned['tags'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

# 5. Export Artifacts
print("5. Saving artifacts...")
# Save processed DataFrame
with open('model/movie_dict.pkl', 'wb') as f:
    pickle.dump(movies_cleaned.to_dict('records'), f)

# Save the sparse matrix efficiently using Scipy
sp.save_npz('model/tfidf_matrix.npz', tfidf_matrix)

print("Processing complete! Models saved.")

1. Loading dataset...
Initial shape: (1452844, 24)
Shape after filtering quality rows: (122691, 24)
4. Vectorizing tags using TF-IDF...
TF-IDF Matrix Shape: (122691, 25000)
5. Saving artifacts...
Processing complete! Models saved.
